In [ ]:
import os
import pprint
from pathlib import Path

import gnss_tools.signals.gps_l1ca as gps_l1ca
import matplotlib.pyplot as plt
import numpy as np
import yaml

from scipy.signal import ShortTimeFFT
import numpy.fft as fft
import scipy.signal
import scipy.signal.windows as signal_windows
import tqdm

import utils
import utils.collect_metadata_utils as collect_metadata_utils
from utils import bpsk_acquisition, bpsk_correlation, sample_streaming

In [ ]:
# Note: you will need to change the filepath to point to where your data is stored
#  I have mine stored in a "local-data" folder within the project directory
local_data_dir = Path(utils.__file__).parent.parent / "local-data"
collects_dir = local_data_dir / "collects"
available_experiment_filepaths = sorted(collects_dir.iterdir())
print("Available experiments:", ", ".join([str(fp.name) for fp in available_experiment_filepaths]))
experiment_name = available_experiment_filepaths[2].name  # change index to select different experiment
experiment_dir = collects_dir / experiment_name
# Note: I keep a metadata.yml file in each experiment directory to keep track of sample metadata
#  and what are the available collect filenames.  You can either create your own metadata.yml,
#  or modify the code below to directly chose your data filepath and set sample parameters.
metadata_filepath = experiment_dir / "metadata.yml"
metadata = collect_metadata_utils.load_experiment_metadata_from_file(metadata_filepath, print_summary=True)

collect_id = metadata.collect_ids[1]
band_id = metadata.band_ids[1]

collect_config = metadata.collects[collect_id]
channel_id = collect_config.channel_config_id
band_config = metadata.band_configurations[band_id]
channel_config = metadata.channel_configurations[channel_id]

inter_freq_l1_hz = band_config.inter_freq
samp_rate = channel_config.samp_rate
sample_params = channel_config.sample_params
collect_filepath = experiment_dir / collect_config.filename

In [ ]:
fig = plt.figure(figsize=(12, 2), dpi=150)
collect_metadata_utils.plot_receiver_channel_bands(fig, metadata)

In [ ]:
buffer_duration_ms = 40
buffer_size_samples = int(samp_rate * buffer_duration_ms / 1e3)
buffer_size_bytes = sample_streaming.compute_sample_array_size_bytes(
    buffer_size_samples, sample_params.bit_depth, sample_params.is_complex
)
byte_buffer = bytearray(buffer_size_bytes)
sample_buffer = np.zeros(buffer_size_samples, dtype=np.complex64)

with open(collect_filepath, "rb") as f:
    f.readinto(byte_buffer)

sample_streaming.convert_to_complex64_samples(
    byte_buffer,
    sample_buffer,
    sample_params
)
baseband_sample_buffer = np.zeros(buffer_size_samples, dtype=np.complex64)
sample_streaming.mixdown_samples(
    sample_buffer,
    baseband_sample_buffer,
    samp_rate,
    0.0,
    inter_freq_l1_hz
)

baseband_sample_buffer -= np.mean(baseband_sample_buffer)
# baseband_sample_buffer *= np.exp(-1j * np.angle(np.mean(np.exp(1j * np.angle(baseband_sample_buffer)))))

In [ ]:
fig = plt.figure(figsize=(10, 4))
axes = fig.subplots(1, 2, width_ratios=[1.5, 1])
ax1: plt.Axes = axes[0]
ax2: plt.Axes = axes[1]
hist_bins = np.arange(-2**(sample_params.bit_depth-1), 2**(sample_params.bit_depth-1))
ax1.hist(baseband_sample_buffer.real, bins=hist_bins, histtype="stepfilled", color="r", alpha=0.6, align="left", label="Real")
ax1.hist(baseband_sample_buffer.imag, bins=hist_bins, histtype="stepfilled", color="b", alpha=0.6, align="mid", label="Imaginary")
ax1.set_xlabel("Sample Value")
ax1.set_ylabel("Count")
ax1.grid()
ax1.legend()

ax2.scatter(baseband_sample_buffer.real, baseband_sample_buffer.imag, color="k", s=1, alpha=0.01, zorder=1)
ax2.set_axisbelow(True)
ax2.grid()
ax2.set_xlabel("Real")
ax2.set_ylabel("Imaginary")
ax2.legend()
plt.show()

# NOTE: should see normal-looking distribution for raw samples.  But if samples are real, the imaginary component should be all zeros.

In [ ]:
# Plot Welch PSD estimate of the samples
fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(1, 1, 1)
freqs, psd_orig = scipy.signal.welch(
    sample_buffer,
    fs=samp_rate,
    nperseg=4096,
    noverlap=2048,
    window="hann",
    return_onesided=False,
    scaling="density",
)
_, psd = scipy.signal.welch(
    baseband_sample_buffer,
    fs=samp_rate,
    nperseg=4096,
    noverlap=2048,
    window="hann",
    return_onesided=False,
    scaling="density",
)
freqs = np.fft.fftshift(freqs)
psd_orig = np.fft.fftshift(psd_orig)
psd = np.fft.fftshift(psd)
# ax.plot(freqs/1e6, 10*np.log10(psd_orig), color="black")
ax.plot(freqs/1e6, 10*np.log10(psd_orig), color="gray")
ax.plot(freqs/1e6, 10*np.log10(psd), color="black")
ax.set_title("Welch PSD Estimate of Raw Samples")
ax.set_xlabel("Frequency (MHz)")
ax.set_ylabel("Power/Frequency (dB/Hz)")
ax.grid()
plt.show()

In [ ]:
stft_interval_ms = 1000
buffer_skip = stft_interval_ms // buffer_duration_ms
periodogram_nperseg = 4096
periodogram_noverlap = 2048
stft_freq = fft.fftshift(fft.fftfreq(periodogram_nperseg, 1 / samp_rate))
# stft = ShortTimeFFT(hamming(buffer_size_samples), buffer_size_samples, samp_rate)

file_duration_estimate_ms = os.path.getsize(collect_filepath) // buffer_size_bytes * buffer_duration_ms
num_windows = file_duration_estimate_ms // stft_interval_ms
periodogram = np.zeros((num_windows, periodogram_nperseg))

# compute STFT periodogram of the signal
with sample_streaming.FileSampleStream(
        collect_filepath,
        sample_params,
        buffer_size_samples,
    ) as sample_stream:

    sample_buffer_generator = sample_stream.sample_buffer_generator(skip=buffer_skip)
    for i, sample_buffer in enumerate(tqdm.tqdm(sample_buffer_generator, total=num_windows)):
        _, epoch_psd = scipy.signal.welch(
            sample_buffer,
            fs=samp_rate,
            nperseg=periodogram_nperseg,
            noverlap=periodogram_noverlap,
            window="hann",
            return_onesided=False,
            scaling="density",
        )
        periodogram[i, :] = fft.fftshift(epoch_psd)
        
        if i + 1 == num_windows:
            break

In [ ]:
# Plot the periodogram
fig = plt.figure(figsize=(10, 4))
plt.imshow(10 * np.log10(periodogram.T), aspect="auto", origin="lower", extent=[0, num_windows * stft_interval_ms / 1000, stft_freq[0]/1e6, stft_freq[-1]/1e6], interpolation="nearest")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (MHz)")
plt.title("STFT Periodogram")
plt.colorbar(label="Power/Frequency (dB/Hz)")
plt.show()

# Acq. File Testing

In [ ]:
GPS_L1CA_acq_code_params = {
    f"G{prn:02}": bpsk_acquisition.AcqSignalCodeParameters(
        rate_chips_per_sec=gps_l1ca.CODE_RATE,
        length_chips=gps_l1ca.CODE_LENGTH,
        sequence=gps_l1ca.get_GPS_L1CA_code_sequence(prn),
    ) for prn in range(1, 33)
}

acq_config = bpsk_acquisition.AcquisitionConfiguration(
    replica_duration_ms=4,
    num_blocks=8,
    sample_rate=samp_rate,
    min_search_doppler_hz=-5000,
    max_search_doppler_hz=5000,
)

# Compute bin probability of false alarm
# If we treat each delay/Doppler bin as single detection test
# individual probability of false alarm per bin p_fa_bin
# yields overall probability of false alarm p_fa_total
num_doppler_bins = acq_config.num_doppler_bins
num_code_phase_bins = acq_config.replica_length_samples
num_detection_tests = num_doppler_bins * num_code_phase_bins
print(f"Acq. num Doppler bins: {num_doppler_bins}")
print(f"Acq. num code phase bins: {num_code_phase_bins}")
print(f"Number of detection tests per signal: {num_detection_tests}")
p_fa_total = 1e-6
p_fa_bin = 1 - (1 - p_fa_total) ** (1 / num_detection_tests)
print(f"Desired overall P_fa: {p_fa_total:.2e}, per-bin P_fa: {p_fa_bin:.2e}")
print("")

acq_results = bpsk_acquisition.run_acquisition(
    baseband_sample_buffer,
    acq_config,
    GPS_L1CA_acq_code_params,
    prob_false_alaram=p_fa_bin,
    print_progress=True,
    noise_var_method="abscorrvar"
)

In [ ]:
# Plot peak amplitudes
fig = plt.figure(figsize=(10, 4), dpi=150)
ax = fig.add_subplot(1, 1, 1)
all_sig_ids = sorted(acq_results.keys())
all_peak_vals = [
    acq_results[sig_id].normalized_peak_value
    for sig_id in all_sig_ids
]
ax.stem(range(len(all_peak_vals)), all_peak_vals, basefmt=" ")
detection_threshold = acq_results["G01"].detection_threshold
ax.plot([0, 33], [detection_threshold] * 2, color="red", linestyle="--", label="Detection Threshold")
ax.legend()
ax.set_yscale("log")
ax.set_xticks(range(len(all_sig_ids)))
ax.set_xticklabels(all_sig_ids, rotation=45)
ax.set_title("Acquisition Peak Correlation Values")
ax.set_xlabel("Signal ID (PRN)")
ax.set_ylabel("Normalized Peak Correlation Value")
ax.grid()
plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 4), dpi=150)
ax = fig.add_subplot(1, 1, 1)
cmap = plt.get_cmap("tab20b")
for i, (signal_id, acq_result) in enumerate(acq_results.items()):
    color = cmap(i / 32.0)
    ax.plot(
        acq_result.correlation_result.doppler_bins_hz,
        acq_result.correlation_result.correlation_matrix[:, acq_result.peak_code_phase_bin],
        color=color,
        marker="o",
        # label=f"{signal_id} (SNR={acq_result.acq_snr_dB:.1f} dB)",
        label=f"{signal_id}",
    )
ax.set_yscale("log")
ax.set_title("Acquisition Correlation Results")
ax.set_xlabel("Doppler Frequency [Hz]")
ax.set_ylabel("Peak Slice Correlation Magnitude")
ax.legend(ncol=4, fontsize=8)
ax.grid()
plt.show()

In [ ]:
# print acquired signal dopplers
acquired_sig_ids = [acq_result.signal_id for acq_result in acq_results.values() if acq_result.signal_detected]
acquired_dopplers = []
for sig_id, acq_result in acq_results.items():
    if acq_result.normalized_peak_value < 200:
        continue
    acquired_dopplers.append(acq_result.acq_doppler_hz)

In [ ]:
fig = plt.figure(figsize=(10, 6), dpi=150)
ax = fig.add_subplot(111)
sig_id = acquired_sig_ids[2]
acq_result = acq_results[sig_id]
acq_config = acq_result.config
correlation = acq_result.correlation_result.correlation_matrix
num_doppler_bins, num_code_phases = correlation.shape
extent = [0, num_code_phases, acq_config.min_search_doppler_hz, acq_config.max_search_doppler_hz]

peak_doppler_bin = acq_result.peak_doppler_bin
peak_doppler_hz = acq_result.correlation_result.doppler_bins_hz[peak_doppler_bin]
peak_code_phase_bin = acq_result.peak_code_phase_bin
print(f"Peak correlation at Doppler: {peak_doppler_hz} Hz, Code Phase: {peak_code_phase_bin} samples")

im = ax.imshow(
    correlation,
    extent=extent,
    aspect="auto",
    interpolation="nearest",
    cmap="plasma",
    origin="lower",
    # vmin=noise_power,
    vmin=0,
    # vmax=500
)
# ax.hlines(acquired_dopplers, 0, num_code_phases, colors="white", linestyles="--", label="Acquired Dopplers")
# code_phase_window = 30000  # samples
code_phase_window = 150  # samples
ax.set_xlim(peak_code_phase_bin - code_phase_window, peak_code_phase_bin + code_phase_window)
ax.set_xlabel("Code Phase [samples]")
ax.set_ylabel("Doppler Frequency [Hz]")
ax.set_title(f"Delay-Doppler Correlation Map for PRN {int(sig_id[1:])}")
plt.colorbar(im, label="Correlation Magnitude")
plt.show()

In [ ]:
sig_id = acquired_sig_ids[0]
acq_result = acq_results[sig_id]
corr_matrix = acq_result.correlation_result.correlation_matrix

y_noise_mean = np.mean(corr_matrix)
sigma_n = np.sqrt(y_noise_mean / (2 * acq_config.num_blocks))
# sigma_n = np.std(sample_buffer)

# y_noise_var = np.var(corr_matrix)
# sigma_n = np.sqrt(np.sqrt(y_noise_var / (4 * acq_config.num_blocks)))


normalized_corr_matrix = corr_matrix / sigma_n**2
# hist_max_val = np.max(normalized_corr_matrix)
hist_max_val = 120

hist_bins = np.linspace(0, hist_max_val, 100)
hist = np.histogram(normalized_corr_matrix.flatten(), bins=hist_bins)[0]

# plot corr matrix histogram
fig = plt.figure(figsize=(8, 4), dpi=150)
# fig = plt.figure(figsize=(5, 4), dpi=150)
ax = fig.add_subplot(1, 1, 1)
ax.bar(
    hist_bins[:-1],
    hist,
    width=hist_bins[1]-hist_bins[0],
    color="blue",
    alpha=0.7,
)
# add chi-squared distribution overlay
x_vals = np.linspace(0, np.max(normalized_corr_matrix), 1000)
chi2_pdf = scipy.stats.chi2.pdf(x_vals, df=2 * acq_config.num_blocks)
ax.plot(
    x_vals,
    chi2_pdf * np.max(hist) / np.max(chi2_pdf),
    color="red",
    linewidth=2,
    label=f"Chi-squared PDF (df={2 * acq_config.num_blocks})",
)
ax.vlines([acq_result.detection_threshold], ymin=0, ymax=np.max(hist), color="black", linestyle="--", label="Detection Threshold")
ax.legend()
ax.set_yscale("log")
ax.set_ylim(1, 1e6)
ax.set_title(f"Histogram of Correlation Magnitudes for {sig_id}")
ax.set_xlabel("Correlation Magnitude")
ax.set_ylabel("Count")
ax.grid()
ax.set_xlim(0, 120)
plt.show()